MCS 320 Quiz 3 Monday 29 June 2026

# Question 1

Make a fast callable object of
$\displaystyle \frac{x^3 - y + 2}{y^3 - x + 2}$
and draw the expression tree. 

## answer to question 1

In [1]:
from sage.ext.fast_callable import ExpressionTreeBuilder
etb = ExpressionTreeBuilder(vars=['x','y'])
x = etb.var('x')
y = etb.var('y')
q = (x^3 - y + 2)/(y^3 - x + 2)
show(q)

div(add(sub(ipow(v_0, 3), v_1), 2), add(sub(ipow(v_1, 3), v_0), 2))

In [2]:
L0 = LabelledBinaryTree([None, None], label='v_0')
L1 = LabelledBinaryTree([None, None], label='v_1')
L2 = LabelledBinaryTree([None, None], label='2')
L3 = LabelledBinaryTree([None, None], label='3')
Nv0pow3 = LabelledBinaryTree([L0, L3], label='ipow')
ascii_art(Nv0pow3)

  _ipow_
 /      \
v_0      3

In [3]:
Nv1pow2 = LabelledBinaryTree([L1, L2], label='ipow')
ascii_art(Nv1pow2)

  _ipow_
 /      \
v_1      2

In [4]:
N0sub = LabelledBinaryTree([Nv0pow3, L1], label='sub')
ascii_art(N0sub)

     ___sub___
    /         \
  _ipow_       v_1
 /      \      
v_0      3     

In [5]:
N1sub = LabelledBinaryTree([Nv1pow2, L0], label='sub')
ascii_art(N1sub)

     ___sub___
    /         \
  _ipow_       v_0
 /      \      
v_1      2     

In [6]:
N0add = LabelledBinaryTree([N0sub, L2], label='add')
ascii_art(N0add)

          ____add_____
         /            \
     ___sub___         2
    /         \        
  _ipow_       v_1     
 /      \              
v_0      3             

In [7]:
N1add = LabelledBinaryTree([N1sub, L2], label='add')
ascii_art(N0add)

          ____add_____
         /            \
     ___sub___         2
    /         \        
  _ipow_       v_1     
 /      \              
v_0      3             

In [8]:
tree = LabelledBinaryTree([N0add, N1add], label='div')
ascii_art(tree)

                ___________div____________
               /                          \
          ____add_____                 ____add_____
         /            \               /            \
     ___sub___         2          ___sub___         2
    /         \                  /         \        
  _ipow_       v_1             _ipow_       v_0     
 /      \                     /      \              
v_0      3                   v_1      2             

# Question 2

Let  

In [9]:
s = lambda n: float(sum([exp(-(k/n)**2)*sin(2*k/n) for k in range(n)])/n)

be a function to approximate 
$\displaystyle \int_{0}^{1} \exp(-x^2) \sin(2x) dx$.

1. Time the execution of ``s`` for $n = 10000$.
      
   Explain why ``s`` is inefficient.

2. Apply vectorization to improve the efficiency.  Verify the correctness.

   Time the execution of the vectorized function for $n = 10000$,
   compare with timings of ``s``.

## answer to question 2

In [10]:
timeit('s(10000)')

5 loops, best of 3: 813 ms per loop

The function ``s`` is inefficient because the conversion to ``float`` happens only at the very end, whereas all 10000 ``k/n`` are computed exactly.

In [11]:
def vs(n):
    from numpy import linspace, exp, sin, sum
    x = linspace(0, 1-1.0/n, n)
    return sum(exp(-x**2)*sin(2*x))/n

Let us first verify the correctness.

In [12]:
s1 = s(10000)
s1

0.4759702501286586

In [13]:
s2 = vs(10000)
s2

0.4759702501286586

In [14]:
abs(s1 - s2)

0.0

The difference is of the order of machine precision.

In [15]:
timeit('vs(10000)')

625 loops, best of 3: 106 μs per loop

The speedup is remarkable: we went from about 800 milliseconds to 10 microseconds: one magnitude of order difference.